### Multimodel Agents

In [13]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from ipywidgets import FileUpload
from IPython.display import display
import sounddevice as sd
from scipy.io.wavfile import write
from tqdm import tqdm
from pprint import pprint
import base64
import time
import io

#### Model

In [2]:
model = f"bedrock:us.anthropic.claude-opus-4-6-v1"

#### Text Agent

In [ ]:
agent = create_agent(
    model,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

for m in response['messages']:
    m.pretty_print()

# Selenopolis: Capital of The Moon

**Founded:** 2147 CE
**Population:** 2.3 million (as of 2289)
**Location:** Shackleton Crater, Lunar South Pole

---

## Overview

Selenopolis — from the Greek goddess of the Moon, Selene — sprawls beneath a vast geodesic crystalline dome nestled within the permanently shadowed walls of Shackleton Crater. Its towers rise like silver needles from the regolith, connected by enclosed skywalks that shimmer with bioluminescent lighting. The city exists in a state of perpetual twilight, its artificial sky cycling through soft amber dawns and violet dusks to maintain the circadian rhythms of its inhabitants.

## Districts

- **The Aether Quarter** — The governmental heart, home to the Lunar Parliament and the Hall of Tides, where representatives from all lunar settlements convene beneath a ceiling that projects real-time views of Earth.
- **Deepwell** — Built into mining shafts below the crater floor, this working-class district houses ice miners who extrac

#### Image Agent

In [4]:
uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [6]:
print(uploader.value)

({'name': 'moon-png-44662.png', 'type': 'image/png', 'size': 1423653, 'content': <memory at 0x110bb4ac0>, 'last_modified': datetime.datetime(2026, 3, 9, 23, 2, 45, 947000, tzinfo=datetime.timezone.utc)},)


#### Base 64 Encoding

In [7]:
# getting the first and only photo
uploaded_file = uploader.value[0]

# accessing the file content
content = uploaded_file["content"]

# convertinng content to bytes
img_bytes = bytes(content)

# base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

#### Invoking Multimodel Agent

In [ ]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this image. What is it?"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

for m in response['messages']:
    m.pretty_print()

# The Moon

This is a high-resolution, black-and-white photograph of **Earth's Moon** in its **full phase**, set against a white background. The near side — the hemisphere permanently facing Earth — is displayed in stunning detail.

You can clearly see:

- **Maria** (the dark, smooth regions) — ancient basaltic lava plains such as *Mare Imbrium*, *Mare Serenitatis*, *Mare Tranquillitatis*, and *Oceanus Procellarum*
- **Highlands** — the brighter, heavily cratered terrain that makes up the older surface
- **Ray craters** — particularly the brilliant splash of rays emanating from **Tycho** (lower center) and **Copernicus** (center-left), where relatively recent impacts sent bright ejecta streaking across the surface
- Countless **impact craters** of varying sizes, a record of billions of years of bombardment

---

Now, as a science fiction writer, I can't help but look at this image and see something else entirely: **real estate**. Beneath those dark maria lie lava tubes large enough to 

#### Audio Agent

In [14]:
# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.53it/s]


Done.


In [ ]:
agent = create_agent(
    model="bedrock:us.amazon.nova-pro-v1:0",
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

for m in response['messages']:
    m.pretty_print()

UnboundLocalError: cannot access local variable 'formatted_block' where it is not associated with a value